### Train a LSTM for sequence prediction


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42

In [4]:
# Prepare data
df = pd.read_csv('tf_bind_8-SIX6_REF_R1/dataset.csv')

x = df.iloc[:, :-2].values
# one hot encode the sequences
x_tensor = tf.one_hot(x, depth=4)
x = x_tensor.numpy()

y = df.iloc[:, -1].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.8, random_state=SEED)

print(f'Training samples: {x_train.shape[0]}')
print(f'Testing samples: {x_test.shape[0]}')

Training samples: 13158
Testing samples: 52634


In [ ]:
# split sequence into input and output -> input: 6 digits, output: 2 digits
x, y = x_train[:, :-2], x_train[:, -2]
print(x.shape, y.shape)

(13158, 6, 4) (13158, 4)


In [23]:
model = Sequential()
model.add(LSTM(100, input_shape = (x.shape[1:])))
model.add(Dense(4, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 100)            │        42,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 4)              │           404 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,404 (165.64 KB)

 Trainable params: 42,404 (165.64 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(x, y, epochs=100, verbose=1)

Epoch 1/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.2498 - loss: 1.3881
Epoch 2/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.2475 - loss: 1.3875
Epoch 3/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.2579 - loss: 1.3864
Epoch 4/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.2565 - loss: 1.3865
Epoch 5/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.2561 - loss: 1.3860
Epoch 6/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.2522 - loss: 1.3862
Epoch 7/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.2499 - loss: 1.3862
Epoch 8/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.2648 - loss: 1.3860
Epoch 9/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.2568 - loss: 1.3861
Epoch 10/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.2587 - loss: 1.3858
Epoch 11/100
412/412 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.2593 - loss: 1.3859
Epoch 12/100
412/412 ━━━━━━━━

In [31]:
# save trained model
model.save('lstm_sequence_model.h5')

In [30]:
def generate_sequence(seed_sequence, next_chars, max_sequence_len):
    for _ in range(next_chars):
        token_list = tf.one_hot(seed_sequence, depth=4)
        token_list = tf.reshape(token_list, (1, token_list.shape[0], token_list.shape[1]))
        predicted = model.predict(token_list, verbose=0)
        predicted_char = tf.argmax(predicted, axis=1).numpy()[0]
        seed_sequence.append(predicted_char)
        if len(seed_sequence) > max_sequence_len:
            seed_sequence = seed_sequence[1:]
    return seed_sequence


print("Sequence generation example:")


seed_seq = [0]
number_sequences = 20
sequences = []
for _ in range(number_sequences):
    
    generated_seq = generate_sequence(seed_seq, next_chars=6, max_sequence_len=8)
    sequences.append(generated_seq)
    seed_seq = generated_seq[:-1]  # update seed sequence

# find out similarity of sequences
for i, seq in enumerate(sequences):
    similarities = []
    for j, other_seq in enumerate(sequences):
        if i != j:
            similarity = sum([1 for a, b in zip(seq, other_seq) if a == b]) / len(seq)
            similarities.append(similarity)
    avg_similarity = sum(similarities) / len(similarities)
    print(f"Sequence {i}: {seq}, Average similarity to other sequences: {avg_similarity:.2f}")

Sequence generation example:
Sequence 0: [0, 3, 1, 1, 3, 3, 3], Average similarity to other sequences: 0.28
Sequence 1: [3, 3, 3, 2, 1, 2, 1, 0], Average similarity to other sequences: 0.24
Sequence 2: [2, 1, 0, 3, 1, 2, 3, 0], Average similarity to other sequences: 0.25
Sequence 3: [2, 3, 2, 3, 0, 0, 0, 2], Average similarity to other sequences: 0.24
Sequence 4: [0, 0, 0, 1, 3, 0, 3, 1], Average similarity to other sequences: 0.29
Sequence 5: [0, 3, 3, 1, 1, 0, 0, 3], Average similarity to other sequences: 0.28
Sequence 6: [0, 0, 3, 3, 2, 2, 1, 3], Average similarity to other sequences: 0.25
Sequence 7: [2, 1, 3, 0, 3, 1, 3, 3], Average similarity to other sequences: 0.24
Sequence 8: [1, 3, 1, 1, 3, 1, 0, 1], Average similarity to other sequences: 0.22
Sequence 9: [1, 0, 1, 1, 0, 3, 3, 1], Average similarity to other sequences: 0.20
Sequence 10: [3, 3, 0, 0, 3, 2, 1, 2], Average similarity to other sequences: 0.24
Sequence 11: [2, 1, 2, 1, 2, 3, 1, 2], Average similarity to other sequ